In [1]:
# Run this diagnostic cell
import sys
from pathlib import Path

monolith_root = Path.home() / "project" / "monolith"
print(f"Monolith root: {monolith_root}")
print(f"Exists: {monolith_root.exists()}")

src_dir = monolith_root / "src"
print(f"src/ exists: {src_dir.exists()}")

data_dir = src_dir / "data"
print(f"src/data/ exists: {data_dir.exists()}")

utkface_file = data_dir / "utkface_dataset.py"
print(f"utkface_dataset.py exists: {utkface_file.exists()}")

# List all files in src/data/
if data_dir.exists():
    print("\nFiles in src/data/:")
    for f in data_dir.iterdir():
        print(f"  - {f.name}")

Monolith root: /home/govindu.srimaan/project/monolith
Exists: True
src/ exists: True
src/data/ exists: True
utkface_dataset.py exists: True

Files in src/data/:
  - adult_dataset.py
  - texas_dataset.py
  - utkface_dataset.py
  - __init__.py
  - __pycache__
  - base_dataset.py


## 1. Imports & Environment Setup

In [2]:
import sys, os, gc, time, random, json, math
from pathlib import Path
from datetime import datetime
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset, Dataset, random_split

MONOLITH_ROOT = Path.home() / "project" / "monolith"
sys.path.insert(0, str(MONOLITH_ROOT))

from src.config import get_utkface_config
from src.data.utkface_dataset import UTKFaceExperiment, TransformSubset, RACE_MAP, RACE_NAMES
from src.models.resnet18 import build_resnet18
from src.trainers.dp_dro_trainer import DPDROTrainer
from src.audit.auditor import SteinkeAuditor
from src.audit.subgroup_analyzer import SubgroupAnalyzer
from src.checkpoint.manager import CheckpointManager, RunDecision
from src.utils.metrics import convert_to_serializable

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

SEED = 2024
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("✅ Environment ready.")

Device: cuda
✅ Environment ready.


## 2. Configuration & Decision Engine 

In [3]:
CFG = get_utkface_config(noise_multiplier=1.1, dp_enabled=True, dro_enabled=True)
CFG.data_dir = MONOLITH_ROOT / "data" / "UTKFace"

# FIX: Prevent DRO divergence
CFG.dro.alpha = 150.0
CFG.dro.eta = 0.001
CFG.dro.lambda_init = 0.0005
CFG.experiment_name = "utkface_main_run"

output_dir = CFG.get_output_dir()
CKPT = CheckpointManager(output_dir, experiment_name=CFG.experiment_name)

temp_model = build_resnet18(num_outputs=1)
decision, reason = CKPT.decide_action(CFG.to_dict(), temp_model)
CKPT.print_decision(decision, reason)
del temp_model; gc.collect(); torch.cuda.empty_cache()


  DECISION: STARTING FRESH RUN
  Reason: Could not determine resume point. Starting fresh.



## 3. Data Loading & Canary Setup

In [4]:
experiment_ds = UTKFaceExperiment(CFG.data_dir)
full_ds = experiment_ds.load()
N = len(full_ds)
k_target = CFG.audit.k
k = max(10, (N - 100) // 2) if N < 2 * k_target + 100 else k_target

saved_split = CKPT.load_data_split()
if saved_split is not None and decision != RunDecision.START_FRESH:
    print("📂 Reusing saved data split.")
    train_idx = list(saved_split['train_idx']); val_idx = list(saved_split['val_idx'])
    canary_in_pool = list(saved_split['canary_in_pool']); canary_out_pool = list(saved_split['canary_out_pool'])
    membership = saved_split['membership']
else:
    print("🆕 Creating new data split.")
    audit_rng = np.random.default_rng(CFG.audit.seed)
    all_indices = list(range(N)); audit_rng.shuffle(all_indices)
    canary_in_pool = all_indices[:k]; canary_out_pool = all_indices[k:2*k]
    remaining_indices = all_indices[2*k:]
    n_val = int(len(remaining_indices) * CFG.train.val_frac)
    gen = torch.Generator().manual_seed(CFG.train.seed)
    train_idx, val_idx = random_split(remaining_indices, [len(remaining_indices) - n_val, n_val], generator=gen)
    train_idx = list(train_idx); val_idx = list(val_idx)
    membership = torch.tensor(audit_rng.integers(0, 2, size=k).astype(bool), dtype=torch.bool)
    CKPT.save_data_split({'train_idx': train_idx, 'val_idx': val_idx, 'canary_in_pool': canary_in_pool, 'canary_out_pool': canary_out_pool, 'membership': membership})

train_ds = TransformSubset(full_ds, train_idx, experiment_ds.get_train_transform())
val_ds = TransformSubset(full_ds, val_idx, experiment_ds.get_val_transform())

class CanaryDataset:
    def __init__(self, base_ds, in_idx, out_idx, membership, transform, rng, val_indices):
        self.base_ds = base_ds; self.membership = membership; self.k = len(membership)
        in_idx = list(in_idx); out_idx = list(out_idx)
        mem_list = membership.tolist() if hasattr(membership, 'tolist') else list(membership)
        self.image_indices = [in_idx[i] if mem_list[i] else out_idx[i] for i in range(self.k)]
        val_list = list(val_indices); held_out_ages = [base_ds.samples[i]['age'] for i in val_list]
        rng.shuffle(held_out_ages)
        self.shuffled_ages = torch.tensor(held_out_ages[:self.k], dtype=torch.float32)
        self.canary_images = [transform(base_ds.get_pil_image(idx)) for idx in self.image_indices]
    def get_in_dataset(self):
        mem_list = self.membership.tolist() if hasattr(self.membership, 'tolist') else list(self.membership)
        in_idxs = [i for i, m in enumerate(mem_list) if m]
        return _InCanarySubset(self, in_idxs), in_idxs
    def get_all_canary_batch(self): return torch.stack(self.canary_images).to(device), self.shuffled_ages.to(device)

class _InCanarySubset(Dataset):
    def __init__(self, cd, in_idxs): self.cd = cd; self.in_idxs = in_idxs
    def __len__(self): return len(self.in_idxs)
    def __getitem__(self, i): return self.cd.canary_images[self.in_idxs[i]].clone(), self.cd.shuffled_ages[self.in_idxs[i]], torch.tensor(-1, dtype=torch.long), torch.tensor(-1, dtype=torch.long)

audit_rng = np.random.default_rng(CFG.audit.seed)
canary_ds = CanaryDataset(full_ds, canary_in_pool, canary_out_pool, membership, experiment_ds.get_canary_transform(), audit_rng, val_idx)
in_canary_ds, _ = canary_ds.get_in_dataset()
augmented_train_ds = ConcatDataset([train_ds, in_canary_ds])
val_loader = DataLoader(val_ds, batch_size=CFG.train.batch_size, shuffle=False, num_workers=CFG.train.num_workers, pin_memory=True)
print("✅ Canary dataset ready.")

  UTKFace loaded: 33483 samples
🆕 Creating new data split.
✅ Canary dataset ready.


## 4. Training Loop Wrapper

In [5]:
def run_training_phase(config, phase_name, is_dro, resume_from_epoch=0):
    k = len(membership)
    model = build_resnet18(num_outputs=1)
    # FIX: Save initial params on the correct device
    initial_params = {name: p.detach().clone().to(device) for name, p in model.named_parameters()}
    
    trainer = DPDROTrainer(config=config, device=device, model=model, full_train_dataset=train_ds, train_indices=list(range(len(train_ds))), train_transform=None)
    train_loader = DataLoader(augmented_train_ds, batch_size=config.train.batch_size, shuffle=True, num_workers=config.train.num_workers, pin_memory=True)
    train_loader = trainer.prepare(train_loader)
    
    start_epoch = 1; history = {'train_loss': [], 'overall_mae': [], 'gender_gap': [], 'race_gap': [], 'epsilon': [], 'lambda_t': []}
    if resume_from_epoch > 0:
        ckpt = CKPT.load_training_checkpoint(phase_name, epoch=resume_from_epoch)
        if ckpt:
            trainer.get_base_model().load_state_dict(ckpt['model_state_dict'])
            trainer.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            start_epoch = ckpt['epoch'] + 1; history = ckpt['history']; trainer.set_dro_state(ckpt.get('dro_state', {}))
            
    if start_epoch == 1:
        with torch.no_grad():
            c_imgs, c_ages = canary_ds.get_all_canary_batch()
            initial_losses = (trainer.get_base_model()(c_imgs) - c_ages).abs().cpu().numpy()
    else:
        initial_losses = CKPT.load_training_checkpoint(phase_name, epoch=resume_from_epoch).get('extra', {}).get('initial_losses')
        if initial_losses is None:
            with torch.no_grad(): initial_losses = (trainer.get_base_model()(canary_ds.get_all_canary_batch()[0]) - canary_ds.get_all_canary_batch()[1]).abs().cpu().numpy()

    print(f'\n[{phase_name}] Training epochs {start_epoch} to {config.train.epochs}...')
    start_time = time.time()
    
    try:
        for epoch in range(start_epoch, config.train.epochs + 1):
            if is_dro and epoch > config.dro.warmup_epochs:
                dro_indices = trainer.get_dro_epoch_indices(epoch)
                if dro_indices is not None:
                    sub_train_indices = [train_idx[i] for i in dro_indices]
                    sub_train_ds = TransformSubset(full_ds, sub_train_indices, experiment_ds.get_train_transform())
                    sub_augmented_ds = ConcatDataset([sub_train_ds, in_canary_ds])
                    # FIX: Just swap loader, do NOT re-wrap Opacus
                    train_loader = DataLoader(sub_augmented_ds, batch_size=config.train.batch_size, shuffle=True, num_workers=config.train.num_workers, pin_memory=True)
            else:
                train_loader = DataLoader(augmented_train_ds, batch_size=config.train.batch_size, shuffle=True, num_workers=config.train.num_workers, pin_memory=True)
                
            epoch_loss = trainer.train_epoch(train_loader)
            if is_dro and epoch > config.dro.warmup_epochs: trainer.update_dro(DataLoader(train_ds, batch_size=256, shuffle=False, num_workers=2))
            
            fair_metrics = evaluate_fairness(trainer.get_base_model(), val_loader)
            history['train_loss'].append(epoch_loss); history['overall_mae'].append(fair_metrics['overall_mae'])
            history['gender_gap'].append(fair_metrics['gender_gap']); history['race_gap'].append(fair_metrics['race_gap'])
            history['epsilon'].append(trainer.get_epsilon()); history['lambda_t'].append(trainer.lambda_t if is_dro else 0.0)
            
            if epoch % 5 == 0 or epoch == start_epoch:
                print(f'  [{phase_name}] Ep {epoch:2d} | L={epoch_loss:.3f} | Gap={fair_metrics["gender_gap"]:.2f} | ε={history["epsilon"][-1]:.2f} | λ={history["lambda_t"][-1]:.6f} | {time.time()-start_time:.0f}s')
            if epoch % config.train.checkpoint_every_n_epochs == 0 or epoch == config.train.epochs:
                CKPT.save_training_checkpoint(phase=phase_name, epoch=epoch, model=trainer.model, optimizer=trainer.optimizer, history=history, dro_state=trainer.get_dro_state(), extra={'initial_losses': initial_losses})
    except KeyboardInterrupt:
        CKPT.save_training_checkpoint(phase=phase_name, epoch=epoch, model=trainer.model, optimizer=trainer.optimizer, history=history, dro_state=trainer.get_dro_state(), extra={'initial_losses': initial_losses})
        return None
        
    with torch.no_grad(): final_losses = (trainer.get_base_model()(canary_ds.get_all_canary_batch()[0]) - canary_ds.get_all_canary_batch()[1]).abs().cpu().numpy()
    
    print(f"  Computing White-Box scores...")
    # FIX: Create a completely fresh model to strip Opacus backward hooks
    wb_model = build_resnet18(num_outputs=1).to(device)
    wb_model.load_state_dict(trainer.get_base_model().state_dict())
    wb_model.eval()
    
    final_params = {name: p.detach() for name, p in wb_model.named_parameters()}
    delta_w = torch.cat([(final_params[name] - initial_params[name]).view(-1) for name in final_params])
    
    c_imgs, c_ages = canary_ds.get_all_canary_batch()
    wb_scores = np.zeros(k, dtype=np.float64)
    for s in range(0, k, 64):
        e = min(s + 64, k)
        sub_imgs = c_imgs[s:e].clone().detach().requires_grad_(True)
        preds = wb_model(sub_imgs)
        losses = (preds - c_ages[s:e]).abs().sum()
        grads = torch.autograd.grad(losses, list(wb_model.parameters()))
        grad_flat = torch.cat([g.view(-1) for g in grads])
        clip_f = min(1.0, config.dp.max_grad_norm / (grad_flat.norm().item() + 1e-8))
        wb_scores[s:e] = torch.dot(delta_w, grad_flat * clip_f).cpu().numpy()
        
    del wb_model
    torch.cuda.empty_cache()
        
    return {'model': trainer.get_base_model(), 'bb_scores': initial_losses - final_losses, 'wb_scores': wb_scores, 'history': history, 'epsilon_spent': history['epsilon'][-1], 'final_metrics': fair_metrics}

def evaluate_fairness(model, loader):
    model.eval(); gender_loss = {0: 0.0, 1: 0.0}; gender_count = {0: 0, 1: 0}
    race_loss = {r: 0.0 for r in RACE_MAP}; race_count = {r: 0 for r in RACE_MAP}
    total_loss = 0.0; total_count = 0
    with torch.no_grad():
        for imgs, ages, genders, races in loader:
            imgs, ages = imgs.to(device), ages.to(device); abs_err = (model(imgs) - ages).abs().cpu()
            valid = (genders >= 0); total_loss += abs_err[valid].sum().item(); total_count += valid.sum().item()
            for g in (0, 1):
                mask = (genders == g) & valid
                if mask.any(): gender_loss[g] += abs_err[mask].sum().item(); gender_count[g] += mask.sum().item()
            for r in RACE_MAP:
                mask = (races == r) & valid
                if mask.any(): race_loss[r] += abs_err[mask].sum().item(); race_count[r] += mask.sum().item()
    male_mae = gender_loss[0] / max(gender_count[0], 1); female_mae = gender_loss[1] / max(gender_count[1], 1)
    race_maes = [race_loss[r] / max(race_count[r], 1) for r in RACE_MAP]
    return {'overall_mae': total_loss / max(total_count, 1), 'male_mae': male_mae, 'female_mae': female_mae, 'gender_gap': abs(male_mae - female_mae), 'race_gap': max(race_maes) - min(race_maes)}

print("✅ Training loop defined.")

✅ Training loop defined.


## 5. Execute ERM & DRO

In [6]:
print('\n' + '═' * 60 + '\n  PHASE 1: DP-SGD + ERM (Baseline)\n' + '═' * 60)
cfg_erm = get_utkface_config(noise_multiplier=CFG.dp.noise_multiplier, dp_enabled=CFG.dp.enabled, dro_enabled=False)
cfg_erm.data_dir = CFG.data_dir; cfg_erm.dro = CFG.dro
erm_ckpt = CKPT.load_training_checkpoint('erm')
resume_epoch = erm_ckpt['epoch'] if erm_ckpt and decision in [RunDecision.RESUME_TRAINING, RunDecision.COMPLETE] else 0
res_erm = run_training_phase(config=cfg_erm, phase_name='erm', is_dro=False, resume_from_epoch=resume_epoch)

print('\n' + '═' * 60 + '\n  PHASE 2: DP-SGD + DRO (Naive Stacking)\n' + '═' * 60)
cfg_dro = get_utkface_config(noise_multiplier=CFG.dp.noise_multiplier, dp_enabled=CFG.dp.enabled, dro_enabled=True)
cfg_dro.data_dir = CFG.data_dir; cfg_dro.dro = CFG.dro
dro_ckpt = CKPT.load_training_checkpoint('dro')
resume_epoch = dro_ckpt['epoch'] if dro_ckpt and decision == RunDecision.COMPLETE else 0
res_dro = run_training_phase(config=cfg_dro, phase_name='dro', is_dro=True, resume_from_epoch=resume_epoch)


════════════════════════════════════════════════════════════
  PHASE 1: DP-SGD + ERM (Baseline)
════════════════════════════════════════════════════════════


/home/govindu.srimaan/mlenv/lib/python3.10/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(



[erm] Training epochs 1 to 25...
  [erm] Ep  1 | L=18.864 | Gap=2.74 | ε=0.21 | λ=0.000000 | 43s
  [erm] Ep  5 | L=16.220 | Gap=2.29 | ε=0.46 | λ=0.000000 | 213s
  [erm] Ep 10 | L=15.151 | Gap=1.59 | ε=0.65 | λ=0.000000 | 433s
  [erm] Ep 15 | L=13.849 | Gap=0.51 | ε=0.81 | λ=0.000000 | 650s
  [erm] Ep 20 | L=13.028 | Gap=0.12 | ε=0.94 | λ=0.000000 | 869s
  [erm] Ep 25 | L=12.466 | Gap=0.37 | ε=1.05 | λ=0.000000 | 1089s
  Computing White-Box scores...

════════════════════════════════════════════════════════════
  PHASE 2: DP-SGD + DRO (Naive Stacking)
════════════════════════════════════════════════════════════

[dro] Training epochs 1 to 25...
  [dro] Ep  1 | L=19.204 | Gap=2.83 | ε=0.21 | λ=0.000500 | 41s
  [dro] Ep  5 | L=17.263 | Gap=2.50 | ε=0.39 | λ=0.003333 | 273s
  [dro] Ep 10 | L=18.955 | Gap=1.81 | ε=0.52 | λ=0.003333 | 577s
  [dro] Ep 15 | L=20.067 | Gap=0.82 | ε=0.62 | λ=0.003333 | 877s
  [dro] Ep 20 | L=20.237 | Gap=0.01 | ε=0.70 | λ=0.003333 | 1167s
  [dro] Ep 25 | L=20.

## 6. Privacy Audit & Subgroup Vulnerability

In [7]:
print('\n' + '═' * 60 + '\n  PHASE 3: Privacy Audit (Steinke Framework)\n' + '═' * 60)
auditor = SteinkeAuditor(gamma=CFG.audit.gamma)
bb_erm = auditor.audit(res_erm['bb_scores'], membership.numpy())
wb_erm = auditor.audit(res_erm['wb_scores'], membership.numpy())
bb_dro = auditor.audit(res_dro['bb_scores'], membership.numpy())
wb_dro = auditor.audit(res_dro['wb_scores'], membership.numpy())

print('\n' + '═' * 60 + '\n  PHASE 4: Disparate Vulnerability Analysis\n' + '═' * 60)
mem_list = membership.tolist()
canary_image_indices = [canary_in_pool[i] if mem_list[i] else canary_out_pool[i] for i in range(k)]
canary_subgroup_labels = experiment_ds.get_subgroup_labels(canary_image_indices)

analyzer = SubgroupAnalyzer({'race': RACE_NAMES, 'gender': ['Male', 'Female']})
erm_subgroup_results = analyzer.analyze(res_erm['wb_scores'], membership.numpy(), canary_subgroup_labels)
dro_subgroup_results = analyzer.analyze(res_dro['wb_scores'], membership.numpy(), canary_subgroup_labels)
erm_disparities = analyzer.compute_disparity(erm_subgroup_results)
dro_disparities = analyzer.compute_disparity(dro_subgroup_results)


════════════════════════════════════════════════════════════
  PHASE 3: Privacy Audit (Steinke Framework)
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  PHASE 4: Disparate Vulnerability Analysis
════════════════════════════════════════════════════════════


## 7. Save Final Results

In [8]:
final_results = {
    'timestamp': datetime.now().isoformat(), 'config': convert_to_serializable(CFG.to_dict()),
    'erm': {'history': convert_to_serializable(res_erm['history']), 'epsilon_spent': float(res_erm['epsilon_spent']), 'bb_audit': convert_to_serializable(bb_erm), 'wb_audit': convert_to_serializable(wb_erm), 'final_metrics': convert_to_serializable(res_erm['final_metrics']), 'subgroup_auc': convert_to_serializable(erm_subgroup_results)},
    'dro': {'history': convert_to_serializable(res_dro['history']), 'epsilon_spent': float(res_dro['epsilon_spent']), 'bb_audit': convert_to_serializable(bb_dro), 'wb_audit': convert_to_serializable(wb_dro), 'final_metrics': convert_to_serializable(res_dro['final_metrics']), 'final_lambda_t': float(res_dro['history']['lambda_t'][-1]), 'subgroup_auc': convert_to_serializable(dro_subgroup_results)}
}
CKPT.save_final_results(final_results)
metadata = CKPT.load_metadata() or {}; metadata['status'] = 'complete'
metadata['completed_phases'] = list(set(metadata.get('completed_phases', []) + ['audit', 'complete']))
CKPT.save_metadata(metadata)
print("✅ UTKFace Experiment COMPLETE! Saved to:", output_dir / 'checkpoints' / 'final_results.json')

✅ UTKFace Experiment COMPLETE! Saved to: experiments/utkface_main_run/utkface_dro_dp_sigma1.1/checkpoints/final_results.json
